<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/15-choosing-an-algorithm.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 15 — Choosing an Algorithm

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/choosing-an-algorithm/).

An honest bake-off: every algorithm in the book on the three worlds the book built, with training time and prediction time recorded alongside the score.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_validate, StratifiedKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.svm import SVR, SVC
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import PolynomialFeatures

def sigmoid(z): return 1 / (1 + np.exp(-z))
def bicycle(km): return 10 + 3.2 * km + 0.15 * km**2
def rush(hour): return 25 + 12 * np.exp(-(hour - 13)**2 / 2) + 15 * np.exp(-(hour - 20)**2 / 2)

# World one: deliveries, a regression with a curve and two rush hours
rng = np.random.default_rng(1)
n = 1500
km, hour, items = rng.uniform(0.5, 8, n), rng.uniform(10, 23, n), rng.integers(1, 7, n)
minutes = bicycle(km) + rush(hour) - 25 + 1.5 * items + rng.normal(0, 4, n)
X_del = np.column_stack([km, hour, items])

# World two: loan defaults, a classification made of thresholds
def make_loans(n, seed=0):
    rng = np.random.default_rng(seed)
    income, debt = rng.uniform(15, 120, n), rng.uniform(0, 0.8, n)
    missed, age = rng.integers(0, 4, n), rng.uniform(21, 70, n)
    risky = ((income < 40) & (debt > 0.4)) | (missed >= 2)
    y = np.where(risky, rng.uniform(size=n) < 0.7, rng.uniform(size=n) < 0.05).astype(int)
    return np.column_stack([income, debt, missed, age]), y
X_loan, y_loan = make_loans(1500, seed=1)

# World three: the digits, 64 pixels, ten classes
from sklearn.datasets import load_digits
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
print(f'deliveries {X_del.shape}, loans {X_loan.shape}, digits {X_dig.shape}')

## 1. The bake-off

In [ ]:
def bake_off(models, X, y, scoring, cv, higher_is_better=True):
    rows = []
    for name, model in models:
        r = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
        score = r['test_score'].mean() if higher_is_better else -r['test_score'].mean()
        fit_ms = r['fit_time'].mean() * 1000
        start = time.perf_counter()
        model.fit(X, y)
        for _ in range(5):
            model.predict(X[:1000])
        pred_ms = (time.perf_counter() - start) / 5 * 1000
        rows.append((name, score, r['test_score'].std(), fit_ms, pred_ms))
    return rows

def show(rows, metric):
    print(f'{"model":<24} {metric:>9} {"±":>6} {"fit ms":>8} {"predict ms/1k":>14}')
    for name, score, sd, fit_ms, pred_ms in rows:
        print(f'{name:<24} {score:>9.3f} {sd:>6.3f} {fit_ms:>8.0f} {pred_ms:>14.2f}')

regressors = [
    ('always the mean', DummyRegressor()),
    ('linear regression', LinearRegression()),
    ('ridge + degree-3 terms', make_pipeline(PolynomialFeatures(3), StandardScaler(), Ridge(alpha=1))),
    ('k-NN, k=10', make_pipeline(StandardScaler(), KNeighborsRegressor(10))),
    ('decision tree', DecisionTreeRegressor(min_samples_leaf=20, random_state=0)),
    ('random forest', RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)),
    ('gradient boosting', GradientBoostingRegressor(n_estimators=300, max_depth=2, learning_rate=0.1, random_state=0)),
    ('SVR, RBF', make_pipeline(StandardScaler(), SVR(C=30, epsilon=1))),
    ('neural net (64, 64)', make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=2000, early_stopping=True, random_state=0))),
]
print('DELIVERIES: mean absolute error in minutes (lower is better). The floor is about 3.2.')
show(bake_off(regressors, X_del, minutes, 'neg_mean_absolute_error', KFold(5, shuffle=True, random_state=0), higher_is_better=False), 'MAE')

In [ ]:
classifiers = [
    ('always the majority', DummyClassifier(strategy='most_frequent')),
    ('logistic regression', make_pipeline(StandardScaler(), LogisticRegression())),
    ('logistic + degree-2', make_pipeline(PolynomialFeatures(2), StandardScaler(), LogisticRegression(max_iter=2000))),
    ('k-NN, k=10', make_pipeline(StandardScaler(), KNeighborsClassifier(10))),
    ('decision tree, depth 3', DecisionTreeClassifier(max_depth=3, random_state=0)),
    ('random forest', RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)),
    ('gradient boosting', GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=0)),
    ('SVM, RBF', make_pipeline(StandardScaler(), SVC(C=1, gamma='scale'))),
    ('neural net (64, 64)', make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=2000, early_stopping=True, random_state=0))),
]
print('LOANS: accuracy (higher is better). The ceiling set by the label noise is about 0.80.')
loan_rows = bake_off(classifiers, X_loan, y_loan, 'accuracy', StratifiedKFold(5, shuffle=True, random_state=0))
show(loan_rows, 'accuracy')

In [ ]:
print('DIGITS: accuracy over ten classes.')
show(bake_off(classifiers, X_dig, y_dig, 'accuracy', StratifiedKFold(5, shuffle=True, random_state=0)), 'accuracy')

## 2. What a difference of a point is worth

In [ ]:
from sklearn.model_selection import cross_val_score
cv = StratifiedKFold(5, shuffle=True, random_state=0)
a = cross_val_score(make_pipeline(StandardScaler(), SVC(C=1, gamma='scale')), X_dig, y_dig, cv=cv)
b = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1), X_dig, y_dig, cv=cv)
print('fold by fold, SVM  :', np.round(a, 3))
print('fold by fold, forest:', np.round(b, 3))
print(f'difference {a.mean() - b.mean():+.3f}, and the folds disagree by up to {np.max(np.abs(a - b)):.3f}')

## 3. How the ranking changes with the size of the data

In [ ]:
print(f'{"n":>5} {"logistic":>9} {"forest":>8} {"boosting":>9} {"net":>6}')
for n_rows in [100, 300, 1000, 1500]:
    Xs, ys = X_loan[:n_rows], y_loan[:n_rows]
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    scores = [cross_val_score(m, Xs, ys, cv=cv).mean() for m in [
        make_pipeline(StandardScaler(), LogisticRegression()),
        RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
        GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=0),
        make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=2000, early_stopping=True, random_state=0))]]
    print(f'{n_rows:>5} ' + ' '.join(f'{s:>8.3f}' for s in scores))

## 4. The cost of a prediction, at scale

In [ ]:
per_1k = {name: pred for name, _, _, _, pred in loan_rows[1:]}          # the loans timings from section 1
requests_per_day = 50_000_000
print(f'{"model":<24} {"ms per 1k":>10} {"cpu-hours/day at 50M":>22}')
for name, ms in per_1k.items():
    print(f'{name:<24} {ms:>10.2f} {ms / 1000 * requests_per_day / 1000 / 3600:>22.2f}')

## 5. What you can explain

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_loan, y_loan)
print('logistic regression, odds multipliers per standard deviation:',
      dict(zip(['income', 'debt', 'missed', 'age'], np.round(np.exp(lr[-1].coef_[0]), 2).tolist())))
from sklearn.tree import export_text
print(export_text(DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_loan, y_loan),
                  feature_names=['income', 'debt', 'missed', 'age'], decimals=1))
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(X_loan, y_loan)
print('random forest: permutation importance only, no rule to print')

## Try it yourself

1. Add `max_depth=6` gradient boosting to the loans bake-off. Does the extra depth help on a world built from two-way thresholds?
2. Reduce the digits to 20 PCA components before every classifier. Which models speed up most, and which lose accuracy?
3. Give the deliveries world a tenth as much noise. Which models move closest to the new floor, and which were limited by their family rather than the noise?